<a href="https://colab.research.google.com/github/Shantanu-Jadhav/Repo1/blob/main/ALS_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install implicit pandas scipy scikit-learn

In [6]:
from google.colab import files

uploaded = files.upload()

Saving als_interactions.csv to als_interactions (1).csv


In [12]:
import os

print(os.listdir())

['.config', 'als_interactions.csv', 'als_interactions (1).csv', 'sample_data']


In [13]:
df=pd.read_csv("als_interactions.csv")
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate user-product pairs:")
print(df.duplicated(
    subset=["user_id", "product_obj_id"]
).sum())

print("\nInteraction values:")
print(df["interaction_value"].describe())

Missing values:
user_id              0
product_obj_id       0
interaction_value    0
dtype: int64

Duplicate user-product pairs:
0

Interaction values:
count    54934.000000
mean        10.608476
std         26.616785
min          1.000000
25%          1.000000
50%          3.000000
75%          9.000000
max       1348.000000
Name: interaction_value, dtype: float64


In [14]:
import pandas as pd
import numpy as np
import scipy.sparse as sparse

try:
    from implicit.als import AlternatingLeastSquares
except ModuleNotFoundError:
    print("Module 'implicit' not found. Attempting to install it now.")
    !pip install implicit
    # After installation, try importing again. If it still fails, a kernel restart is needed.
    try:
        from implicit.als import AlternatingLeastSquares
    except ModuleNotFoundError:
        print("Installation successful but module still not found. Please restart the Colab runtime (Runtime -> Restart runtime) and re-run all cells.")

from sklearn.model_selection import train_test_split

In [15]:
df = pd.read_csv("als_interactions.csv")

In [16]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate user-product pairs:")
print(df.duplicated(
    subset=["user_id", "product_obj_id"]
).sum())

print("\nInteraction values:")
print(df["interaction_value"].describe())

Missing values:
user_id              0
product_obj_id       0
interaction_value    0
dtype: int64

Duplicate user-product pairs:
0

Interaction values:
count    54934.000000
mean        10.608476
std         26.616785
min          1.000000
25%          1.000000
50%          3.000000
75%          9.000000
max       1348.000000
Name: interaction_value, dtype: float64


In [17]:
user_ids = df["user_id"].unique()
product_ids = df["product_obj_id"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

product_to_index = {
    product_id: index
    for index, product_id in enumerate(product_ids)
}

index_to_user = {
    index: user_id
    for user_id, index in user_to_index.items()
}

index_to_product = {
    index: product_id
    for product_id, index in product_to_index.items()
}

df["user_index"] = df["user_id"].map(user_to_index)
df["product_index"] = df["product_obj_id"].map(product_to_index)

In [18]:
user_item_matrix = sparse.csr_matrix(
    (
        df["interaction_value"].astype(float),
        (df["user_index"], df["product_index"])
    ),
    shape=(len(user_ids), len(product_ids))
)

print("Matrix shape:", user_item_matrix.shape)
print("Non-zero interactions:", user_item_matrix.nnz)

Matrix shape: (1024, 332)
Non-zero interactions: 54934


In [19]:
model = AlternatingLeastSquares(
    factors=32,
    regularization=0.05,
    iterations=20,
    random_state=42
)

model.fit(user_item_matrix)

/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [20]:
def train_test_split_interactions(df, test_size=0.2, random_state=42):
    train_df = pd.DataFrame(columns=df.columns)
    test_df = pd.DataFrame(columns=df.columns)

    for user_id in df['user_id'].unique():
        user_interactions = df[df['user_id'] == user_id]
        if len(user_interactions) > 1:
            train_user, test_user = train_test_split(
                user_interactions,
                test_size=test_size,
                random_state=random_state
            )
            train_df = pd.concat([train_df, train_user])
            test_df = pd.concat([test_df, test_user])
        else:
            train_df = pd.concat([train_df, user_interactions])

    return train_df, test_df

train_df, test_df = train_test_split_interactions(df, test_size=0.2, random_state=42)

In [21]:
train_user_item_matrix = sparse.csr_matrix(
    (
        train_df["interaction_value"].astype(float),
        (train_df["user_index"], train_df["product_index"])
    ),
    shape=(len(user_ids), len(product_ids))
)

test_user_item_matrix = sparse.csr_matrix(
    (
        test_df["interaction_value"].astype(float),
        (test_df["user_index"], test_df["product_index"])
    ),
    shape=(len(user_ids), len(product_ids))
)

print("Train matrix shape:", train_user_item_matrix.shape)
print("Train non-zero interactions:", train_user_item_matrix.nnz)
print("Test matrix shape:", test_user_item_matrix.shape)
print("Test non-zero interactions:", test_user_item_matrix.nnz)

Train matrix shape: (1024, 332)
Train non-zero interactions: 43531
Test matrix shape: (1024, 332)
Test non-zero interactions: 11403


In [22]:
model_trained = AlternatingLeastSquares(
    factors=32,
    regularization=0.05,
    iterations=20,
    random_state=42
)

# Train the model on the training data
model_trained.fit(train_user_item_matrix)

  0%|          | 0/20 [00:00<?, ?it/s]

In [23]:
def calculate_precision_recall(model, train_matrix, test_matrix, k=10):
    precision_scores = []
    recall_scores = []

    n_users = train_matrix.shape[0]

    for user_index in range(n_users):
        # Get actual interactions from the test set for the current user
        actual_interactions = set(test_matrix.indices[test_matrix.indptr[user_index]:test_matrix.indptr[user_index+1]])

        if not actual_interactions:  # Skip users with no interactions in the test set
            continue

        # Get top-k recommendations for the user based on their training interactions
        # filter_already_liked_items=True ensures we don't recommend items the user has already interacted with in the training set
        recommendations = model.recommend(
            user_index,
            train_matrix[user_index:user_index+1], # Pass a single row for the user's interactions
            N=k,
            filter_already_liked_items=True
        )

        if not recommendations: # Skip if no recommendations are generated
            continue

        # The `recommend` method returns a tuple of (item_indices, scores). We need to zip them.
        recommended_items = {item_index for item_index in recommendations[0]}

        # Calculate hits (intersection between recommended and actual)
        hits = len(actual_interactions.intersection(recommended_items))

        # Precision@k: (number of hits) / k
        precision_scores.append(hits / k)

        # Recall@k: (number of hits) / (number of actual interactions)
        recall_scores.append(hits / len(actual_interactions))

    avg_precision = np.mean(precision_scores) if precision_scores else 0
    avg_recall = np.mean(recall_scores) if recall_scores else 0

    return avg_precision, avg_recall

# Calculate Precision@5 and Recall@5
precision_at_5, recall_at_5 = calculate_precision_recall(model_trained, train_user_item_matrix, test_user_item_matrix, k=5)
print(f"Precision@5: {precision_at_5:.4f}")
print(f"Recall@5: {recall_at_5:.4f}")

# Calculate Precision@10 and Recall@10
precision_at_10, recall_at_10 = calculate_precision_recall(model_trained, train_user_item_matrix, test_user_item_matrix, k=10)
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")

Precision@5: 0.4409
Recall@5: 0.2062
Precision@10: 0.3643
Recall@10: 0.3343


In [24]:
# Select a sample user from the test set (e.g., the first user in test_df)
sample_user_id = test_df['user_id'].iloc[0]
sample_user_index = user_to_index[sample_user_id]

# Get top-10 recommendations for this user
# We pass the user's interactions from the training matrix to filter out already liked items
recommendations = model_trained.recommend(
    sample_user_index,
    train_user_item_matrix[sample_user_index],
    N=10,
    filter_already_liked_items=True
)

print(f"Top-10 recommendations for user {sample_user_id} (index {sample_user_index}):")
recommended_products_with_scores = []
# Fix: Iterate by zipping the two arrays (indices and scores) returned by recommendations
for product_index, score in zip(recommendations[0], recommendations[1]):
    product_obj_id = index_to_product[product_index]
    recommended_products_with_scores.append({"product_obj_id": product_obj_id, "score": score})

# Display the recommendations as a DataFrame for better readability
recommended_df = pd.DataFrame(recommended_products_with_scores)
display(recommended_df)

Top-10 recommendations for user 27 (index 0):


,product_obj_id,score
0,428,0.975902
1,422,0.972439
2,423,0.957901
3,387,0.955141
4,392,0.952533
5,425,0.951787
6,385,0.927255
7,388,0.865632
8,389,0.826342
9,397,0.820280
